# Support Vector Machine
SVM is a machine learning algorithm used for classification and, to some extent, regression.

* **Basic idea**: Find a boundary (hyperplane) that best separates the different classes in the input data.
* **Margin maximization**: SVM tries to find the hyperplane such that the distance between the boundary and the nearest points of both classes is as large as possible (these points are called support vectors).
* **Linearly separable data**: If the classes can be separated by a line (2D) or a hyperplane (more dimensions), SVM finds the optimal separating hyperplane.
* **Non-linear data**: Using the kernel trick (e.g. RBF, polynomial), the data can be mapped into a higher dimension, where it is linearly separable.
* **Regularization**: The parameter C affects the trade-off between maximizing the margin and minimizing classification errors on the training data.

* Advantages:
    * Works well for high-dimensional data.
    * Robust against overfitting, especially if the kernel and the C parameter are set correctly.

* Disadvantages:
    * Can be slow on large datasets.
    * Choosing a suitable kernel and parameters requires experimentation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Loading the prepared data from the numpy file

In [ ]:
my_arrays = np.load("iris_numpy.npz")
X = my_arrays['X']
y = my_arrays['y']
X_norm = my_arrays['X_norm']
X_features = my_arrays['X_feature']

Splitting the data into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42)

Loading the encoder and the scaler

In [ ]:
import joblib
scaler=joblib.load('classification_std_scaler.bin')
encoder=joblib.load('classification_encoder.bin')

## Training the model
Creating and training a linear SVM model

In [ ]:
from sklearn import svm

svm_model = svm.SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)

Running the model on the test data

In [ ]:
y_pred = svm_model.predict(X_test)

## Model verification

In [ ]:
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = svm_model, X = X_train, y = y_train)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import seaborn as sns

cf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(cf_matrix, annot=True, xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — SVM')
plt.show()

Score

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

## Model visualization

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

The support vectors are key for SVM. These are the points that determine the boundary.

In [ ]:
svm_model.support_vectors_

We will prepare a function that displays the measured values, the support vectors, and the class membership using color, as well as the class boundaries.

In [ ]:
def SVM_vizualization(svm_model, X, y, title, xlabel, ylabel):
    # Display of decision boundaries
    disp = DecisionBoundaryDisplay.from_estimator(
        svm_model,
        X,
        response_method="predict",
        cmap=plt.cm.coolwarm,
        alpha=0.8,
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    # View points
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, s=20, edgecolors="k")

    # View support vectors - larger circles
    try:
        plt.scatter(svm_model.support_vectors_[:, 0], svm_model.support_vectors_[:, 1], s=40, edgecolors="k", facecolors='none')
    except AttributeError:
        pass
    plt.show()

In [ ]:
SVM_vizualization (svm_model, X_train, y_train, "Linear kernel", "Petal length", "Petal width")

## Other kernels
The SVM regularization parameter C is the sensitivity of the margin to the distance of a point; it defines the penalty if some point is misclassified.

We will try calling SVM with a different kernel.

In [ ]:
C = 1.0

### Linear kernel

In [ ]:
svm_linearsvc=svm.LinearSVC(C=C, max_iter=10000)
svm_linearsvc.fit(X_train, y_train)
SVM_vizualization (svm_linearsvc, X_train, y_train, "LinearSVC (linear kernel)", "Petal length", "Petal width")
print (f"Score: {svm_linearsvc.score(X_test, y_test)}")

### RBF kernel

In [ ]:
svc_rbf = svm.SVC(kernel="rbf", gamma=0.7, C=C, random_state=42)
svc_rbf.fit(X_train, y_train)
SVM_vizualization(svc_rbf, X_train, y_train, "RBF kernel", "Petal length", "Petal width")
print(f"Score: {svc_rbf.score(X_test, y_test)}")

### Polynomial kernel
The polynomial degree is set to 3

In [ ]:
svc_poly = svm.SVC(kernel="poly", degree=3, gamma="auto", C=C, random_state=42)
svc_poly.fit(X_train, y_train)
SVM_vizualization(svc_poly, X_train, y_train, "Poly kernel", "Petal length", "Petal width")
print(f"Score: {svc_poly.score(X_test, y_test)}")

## Hyperparameters
Similar to k-means, SVM has several parameters that affect the model's accuracy. Setting them correctly can be key, but sometimes the parameters are hard to guess.

One approach is to try building SVM models for all combinations of parameters and see which one gives the best result.

A large number of models may be created, so this can take a while.

The scikit library already has a function prepared for this.

In [ ]:
from sklearn.model_selection import GridSearchCV
svc=svm.SVC()
params=[
    {"kernel": ["poly"], "C":[1, 5, 10, 50], "degree":[1,2,3,4], "gamma":[0.1, 0.3, 0.7, 1, 5]},
    {"kernel": ["rbf"], "C":[1, 5, 10, ], "gamma":[0.1, 0.5, 1, 5]},    
]

In [ ]:
clf = GridSearchCV(estimator=svc, param_grid=params, cv=5, scoring='accuracy', verbose=1)
clf.fit(X_train, y_train)

Displaying the score of the best model.

In [ ]:
clf.score(X_test, y_test)

Displaying the input parameters of the best model.

In [ ]:
clf.best_params_

And its visualization

In [ ]:
SVM_vizualization (clf, X_train, y_train, "best SVM", "Petal length", "Petal width")

# Saving the model

In [ ]:
filename = 'svm.bin'
joblib.dump(clf, filename)

# Using the model

We will try to use the model to determine 3 different irises. The input values are given in the original units, the same as the original data.

In [ ]:
iris_measurements = np.array([[5.1, 1.9],[1.0, 0.5],[6.0, 2.5]])

Loading and applying the standard scaler

In [ ]:
scaler=joblib.load('classification_std_scaler.bin')
iris_measurements_norm=scaler.transform(iris_measurements)

Loading the label encoder

In [ ]:
encoder=joblib.load('classification_encoder.bin')
encoder_mapping = dict(enumerate(encoder.classes_))

Loading the SVM model

In [ ]:
loaded_model = joblib.load(filename)

Running the model on the data

In [ ]:
predictions = loaded_model.predict(iris_measurements_norm)
predictions

Displaying the answers

In [ ]:
for prediction in predictions:
    print (encoder_mapping[prediction])